In [3]:
# install packages
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install statsmodels
!pip insatll pandas_datareader
!pip install datetime
!pip install yfinance
!pip install sklearn
!pip install PyPortfolioOpt


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [15 lines of output]
  The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
  rather than 'sklearn' for pip commands.
  
  Here is how to fix this error in the main use cases:
  - use 'pip install scikit-learn' rather than 'pip install sklearn'
  - replace 'sklearn' by 'scikit-learn' in your pip requirements files
    (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
  - if the 'sklearn' package is used by one of your dependencies,
    it would be great if you take some time to track which package uses
    'sklearn' instead of 'scikit-learn' and report it to their issue tracker
  - as a last resort, set the environment variable
    SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
  
  More information is available at
  https://github.com/scikit-learn/sklearn-pypi-package
  [end of output]
  
  note: This error originates f

In [13]:
!pip install pandas-ta-classic

### 1. Download/Load SP500 stocks prices data

In [15]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta_classic as ta
import warnings
warnings.filterwarnings('ignore')

In [39]:
# url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
# tables = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})

# sp500 = tables[0]
# sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
# symbols_list = sp500['Symbol'].unique().tolist()


# end_date = "2026-10-17"
# start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

# df = yf.download(tickers=symbols_list,
#                  start=start_date,
#                  end=end_date,
#                  auto_adjust=False).stack()

# df.index.names = ['date', 'ticker']

# df.columns = df.columns.str.lower()

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-10-19 A        60.784538   64.470001   66.239998   64.239998   65.809998   
           AAPL     51.809418   54.827499   55.314999   54.357498   54.514999   
           ABBV     63.036037   87.970001   89.989998   87.970001   89.260002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      58.352303   67.589996   68.650002   67.230003   68.129997   
...                       ...         ...         ...         ...         ...   
2026-10-16 XYZ      77.889999   77.889999   81.174004   76.570000   80.430000   
           YUM     150.509995  150.509995  152.419998  149.520004  151.979996   
           ZBH      94.980003   94.980003   95.910004   94.239998   95.000000   
           ZBRA    369.010010  369.010010  378.109985  367.489990  378.109985   
           ZTS      79.379997   79.379997   80.589996   78.290001   78.290001   

Price                   volume  
date       ticker               
2018-10-19 A         2418300.0  
           AAPL    132314800.0  
           ABBV      7059400.0  
           ABNB            NaN  
           ABT       7351600.0  
...                        ...  
2026-10-16 XYZ       4986100.0  
           YUM       2020100.0  
           ZBH       1935300.0  
           ZBRA       500700.0  
           ZTS       5570700.0  

[1010024 rows x 6 columns]

### 2. Calculate features and technical indicators for each stock.


- Garman-Klass Volatility
- RSI
- Bolinger Bands
- ATR
- MACD
- Dollar Volume

In [60]:
# df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*(np.log(df['adj close'])-np.log(df['open']))**2
# df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: ta.rsi(close=x, length=20))
# df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: ta.bbands(close=np.log1p(x), length=20).iloc[:,0])
# df['bb_med'] = df.groupby(level=1)['adj close'].transform(lambda x: ta.bbands(close=np.log1p(x), length=20).iloc[:,1])
# df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: ta.bbands(close=np.log1p(x), length=20).iloc[:,2])
# def compute_atr(stock_data):
#     atr = ta.atr(high=stock_data['high'],
#                         low=stock_data['low'],
#                         close=stock_data['close'],
#                         length=14)
#     return atr.sub(atr.mean()).div(atr.std())

# df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# def compute_macd(close):
#     macd = ta.macd(close=close)
#     if macd is None:
#         return pd.Series(np.nan, index=close.index)
#     macd = macd.iloc[:, 0]
#     return macd.sub(macd.mean()).div(macd.std())

# df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

# df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-10-19 A        60.784538   64.470001   66.239998   64.239998   65.809998   
           AAPL     51.809418   54.827499   55.314999   54.357498   54.514999   
           ABBV     63.036037   87.970001   89.989998   87.970001   89.260002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      58.352303   67.589996   68.650002   67.230003   68.129997   
...                       ...         ...         ...         ...         ...   
2026-10-16 XYZ      77.889999   77.889999   81.174004   76.570000   80.430000   
           YUM     150.509995  150.509995  152.419998  149.520004  151.979996   
           ZBH      94.980003   94.980003   95.910004   94.239998   95.000000   
           ZBRA    369.010010  369.010010  378.109985  367.489990  378.109985   
           ZTS      79.379997   79.379997   80.589996   78.290001   78.290001   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2018-10-19 A         2418300.0         -0.001968        NaN       NaN   
           AAPL    132314800.0         -0.000849        NaN       NaN   
           ABBV      7059400.0         -0.046483        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT       7351600.0         -0.009053        NaN       NaN   
...                        ...               ...        ...       ...   
2026-10-16 XYZ       4986100.0          0.001307  50.874015  4.301553   
           YUM       2020100.0          0.000148  59.256857  4.893688   
           ZBH       1935300.0          0.000154  54.212980  4.476078   
           ZBRA       500700.0          0.000177  53.275180  5.880563   
           ZTS       5570700.0          0.000345  63.104377  4.224350   

Price                bb_med   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2018-10-19 A            NaN       NaN       NaN       NaN     146.995249  
           AAPL         NaN       NaN       NaN       NaN    6855.152744  
           ABBV         NaN       NaN       NaN       NaN     444.996603  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     428.982787  
...                     ...       ...       ...       ...            ...  
2026-10-16 XYZ     4.344291  4.387029 -0.716083 -0.047369     388.367326  
           YUM     4.957986  5.022283  0.979104  0.671141     304.045240  
           ZBH     4.521261  4.566445 -0.475293 -0.165656     183.814800  
           ZBRA    5.930078  5.979594  0.142445  0.642174     184.763312  
           ZTS     4.298420  4.372490 -1.512572  0.372466     442.202151  

[1010024 rows x 14 columns]

### 3. Aggreate to monthly level and filter top 150 most liquid stocks for each month
- to reduce training time and experiment with features and strategies, we conver business-daily data to month-end frequency

In [91]:
# last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
#                                                           'high', 'low', 'close']]

# data = pd.concat([df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
#           df.unstack()[last_cols].resample("ME").last().stack('ticker')],
#           axis=1).dropna()

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-11-30 A          187.326245   68.214058         -0.000757  65.536180   
           AAPL      8333.096085   42.334633         -0.001346  34.723705   
           ABBV       482.619596   67.550385         -0.031305  59.191816   
           ABT        390.192289   63.929420         -0.008036  64.809495   
           ACGL        28.335316   27.214716         -0.000698  56.557209   
...                          ...         ...               ...        ...   
2026-10-31 XYZ        356.004985   77.889999          0.001307  50.874015   
           YUM        349.481783  150.509995          0.000148  59.256857   
           ZBH        182.533066   94.980003          0.000154  54.212980   
           ZBRA       172.452887  369.010010          0.000177  53.275180   
           ZTS        325.732428   79.379997          0.000345  63.104377   

                     bb_low    bb_med   bb_high       atr      macd  
date       ticker                                                    
2018-11-30 A       4.085408  4.163147  4.240886 -1.303068  0.688791  
           AAPL    3.699004  3.825202  3.951399 -1.197156 -1.344023  
           ABBV    4.084120  4.155541  4.226963 -0.197430  0.420648  
           ABT     4.082028  4.129483  4.176938 -0.849480  0.343911  
           ACGL    3.308769  3.323610  3.338450 -1.384606  0.002136  
...                     ...       ...       ...       ...       ...  
2026-10-31 XYZ     4.301553  4.344291  4.387029 -0.716083 -0.047369  
           YUM     4.893688  4.957986  5.022283  0.979104  0.671141  
           ZBH     4.476078  4.521261  4.566445 -0.475293 -0.165656  
           ZBRA    5.880563  5.930078  5.979594  0.142445  0.642174  
           ZTS     4.224350  4.298420  4.372490 -1.512572  0.372466  

[47160 rows x 9 columns]

- calculat the 5-year rolling average of dollar volume for each stock before filtering

In [92]:
# data['dollar_volume'] = (data['dollar_volume'].unstack('ticker').rolling(5*12).mean().stack())

# data['dollar_vol_rank'] = data.groupby('date')['dollar_volume'].rank(ascending=False)

# data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-11-30 A                 NaN   68.214058         -0.000757  65.536180   
           AAPL              NaN   42.334633         -0.001346  34.723705   
           ABBV              NaN   67.550385         -0.031305  59.191816   
           ABT               NaN   63.929420         -0.008036  64.809495   
           ACGL              NaN   27.214716         -0.000698  56.557209   
...                          ...         ...               ...        ...   
2026-10-31 XYZ        824.017957   77.889999          0.001307  50.874015   
           YUM        247.576685  150.509995          0.000148  59.256857   
           ZBH        181.894877   94.980003          0.000154  54.212980   
           ZBRA       151.094763  369.010010          0.000177  53.275180   
           ZTS        417.056911   79.379997          0.000345  63.104377   

                     bb_low    bb_med   bb_high       atr      macd  \
date       ticker                                                     
2018-11-30 A       4.085408  4.163147  4.240886 -1.303068  0.688791   
           AAPL    3.699004  3.825202  3.951399 -1.197156 -1.344023   
           ABBV    4.084120  4.155541  4.226963 -0.197430  0.420648   
           ABT     4.082028  4.129483  4.176938 -0.849480  0.343911   
           ACGL    3.308769  3.323610  3.338450 -1.384606  0.002136   
...                     ...       ...       ...       ...       ...   
2026-10-31 XYZ     4.301553  4.344291  4.387029 -0.716083 -0.047369   
           YUM     4.893688  4.957986  5.022283  0.979104  0.671141   
           ZBH     4.476078  4.521261  4.566445 -0.475293 -0.165656   
           ZBRA    5.880563  5.930078  5.979594  0.142445  0.642174   
           ZTS     4.224350  4.298420  4.372490 -1.512572  0.372466   

                   dollar_vol_rank  
date       ticker                   
2018-11-30 A                   NaN  
           AAPL                NaN  
           ABBV                NaN  
           ABT                 NaN  
           ACGL                NaN  
...                            ...  
2026-10-31 XYZ                70.0  
           YUM               283.0  
           ZBH               360.0  
           ZBRA              399.0  
           ZTS               166.0  

[47160 rows x 10 columns]

### 4. Calculate Monthly Returns for different time horizons as features
- to capture time series dynamics that reflect, for example, momentum pattners, we compute historical returns using the method .pct_change(lag), that is, returns over various monthly periods as identified by lags.

In [93]:
# g = df.xs('AAPL', level=1)
def calculate_returns(df):
        
    outlier_cutoff = 0.005
    
    lags = [1, 2, 3, 6, 9, 12]
    
    for lag in lags:
        df[f'return_{lag}m'] = (df['adj close']
                               .pct_change(lag)
                               .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                      upper=x.quantile(1-outlier_cutoff)))
                               .add(1)
                               .pow(1/lag)
                               .sub(1))

    return df

data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2023-10-31 A          193.857437  101.089066          0.000105  36.961639   
           AAPL     10982.728146  168.491150          0.000147  42.578591   
           ABBV       648.398335  127.775520         -0.004193  39.783551   
           ABT        508.735803   88.769867         -0.001040  45.034588   
           ACGL        81.681813   82.423882         -0.000007  62.466130   
...                          ...         ...               ...        ...   
2026-10-31 XYZ        824.017957   77.889999          0.001307  50.874015   
           YUM        247.576685  150.509995          0.000148  59.256857   
           ZBH        181.894877   94.980003          0.000154  54.212980   
           ZBRA       151.094763  369.010010          0.000177  53.275180   
           ZTS        417.056911   79.379997          0.000345  63.104377   

                     bb_low    bb_med   bb_high       atr      macd  \
date       ticker                                                     
2023-10-31 A       4.606318  4.673850  4.741381  0.134700 -1.312470   
           AAPL    5.111468  5.155903  5.200338 -0.319800 -0.964981   
           ABBV    4.855389  4.890920  4.926451 -0.074610 -0.964571   
           ABT     4.452313  4.493487  4.534660  0.022952 -0.764018   
           ACGL    4.344124  4.383791  4.423458  1.029514  0.645462   
...                     ...       ...       ...       ...       ...   
2026-10-31 XYZ     4.301553  4.344291  4.387029 -0.716083 -0.047369   
           YUM     4.893688  4.957986  5.022283  0.979104  0.671141   
           ZBH     4.476078  4.521261  4.566445 -0.475293 -0.165656   
           ZBRA    5.880563  5.930078  5.979594  0.142445  0.642174   
           ZTS     4.224350  4.298420  4.372490 -1.512572  0.372466   

                   dollar_vol_rank  return_1m  return_2m  return_3m  \
date       ticker                                                     
2023-10-31 A                 232.0  -0.073704  -0.075054  -0.052506   
           AAPL                2.0  -0.002570  -0.046596  -0.045194   
           ABBV               54.0  -0.043384  -0.014969  -0.015819   
           ABT                78.0  -0.018344  -0.038782  -0.051256   
           ACGL              410.0   0.087442   0.061963   0.037173   
...                            ...        ...        ...        ...   
2026-10-31 XYZ                70.0   0.059296  -0.025502  -0.013939   
           YUM               283.0   0.103931  -0.006679  -0.004393   
           ZBH               360.0   0.069233  -0.024335   0.004593   
           ZBRA              399.0  -0.011519   0.021630   0.078911   
           ZTS               166.0   0.134473   0.014678   0.008934   

                   return_6m  return_9m  return_12m  
date       ticker                                    
2023-10-31 A       -0.043400  -0.038326   -0.023424  
           AAPL     0.001524   0.019380    0.009510  
           ABBV    -0.007812  -0.001703    0.000275  
           ABT     -0.023927  -0.015550   -0.002129  
           ACGL     0.024257   0.033652    0.034795  
...                      ...        ...         ...  
2026-10-31 XYZ      0.016729   0.028603    0.002115  
           YUM     -0.008141  -0.002010    0.008745  
           ZBH      0.024803   0.010606   -0.003862  
           ZBRA     0.084938   0.051426    0.026613  
           ZTS     -0.058781  -0.047855   -0.047235  

[17881 rows x 16 columns]

### 5. Download Fama-French Factors and Calculate Rolling Factor Betas.
- we will introduct the Fama-French data to estimate the exposure of assets to common risk factors using linear regresstion
- the five Fama-French factors, namely market risk, size, value, operating profitability, and investment have been shown to empirically to explain asset returns in the past and are commonly used to assess the risk/return profile of portfolios. Hence, it is natural to include past factor exposures as financial features in the models
- we can access the historical returns using the pandas-datareader and estimate historical exposures using the RollingOLS rolling linear regression